# Google Analytics 4 to Databricks Pipeline Generator

This notebook generates Databricks Asset Bundle YAML files for GA4 ingestion pipelines using prefix+priority grouping logic.

## How it works:
1. **Input CSV with prefix + priority**: You provide a CSV with GA4 properties grouped by business units (prefix) and priorities
2. **Pipeline Grouping**: Each unique (prefix, priority) combination creates a separate pipeline
3. **YAML Generation**: Generates Databricks Asset Bundle configuration with scheduled jobs
4. **Deploy**: Use Databricks CLI to deploy the pipelines

## Pipeline Naming Convention:
- Pipeline name: `{prefix}_{priority}`
- Example: `business_unit1_01`, `analytics_team_02`, etc.

In [0]:
%pip install pyyaml --quiet

In [0]:
dbutils.library.restartPython()

# Step 1: Verify Input CSV

In [ ]:
%sql
SELECT * FROM read_files(
  '/Volumes/your_catalog/your_schema/your_volume/ga4_input_config.csv',
  format => 'csv',
  header => true
)

# Step 2: Configuration

In [ ]:
from pyspark.sql.functions import col, concat_ws, lpad, coalesce, lit

# Configuration
INPUT_CSV = "/Volumes/your_catalog/your_schema/your_volume/ga4_input_config.csv"
OUTPUT_CONFIG = "/Volumes/your_catalog/your_schema/your_volume/config/ga4_pipeline_config.csv"
DEFAULT_SCHEDULE = "0 */6 * * *"

# Read CSV using Spark
input_df = spark.read.format("csv").option("header", "true").load(INPUT_CSV)

print(f"Loaded {input_df.count()} GA4 properties\n")

# Validate required columns
required_columns = ['source_catalog', 'source_schema', 'tables', 'target_catalog', 'target_schema', 'prefix', 'priority']
missing = [c for c in required_columns if c not in input_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Generate pipeline configuration with prefix+priority grouping
config_df = input_df \
    .withColumn("schedule", coalesce(col("schedule"), lit(DEFAULT_SCHEDULE))) \
    .withColumn("priority_padded", lpad(col("priority"), 2, "0")) \
    .withColumn("pipeline_group", concat_ws("_", col("prefix"), col("priority_padded")))

# Show summary
print("Pipeline Configuration Summary:")
print(f"Total properties: {config_df.count()}")
print(f"Unique pipeline groups: {config_df.select('pipeline_group').distinct().count()}\n")

print("Pipeline Groups:")
config_df.groupBy("pipeline_group").count().orderBy("pipeline_group").show(truncate=False)

# Save configuration to volume
config_df.write.format("csv").option("header", "true").mode("overwrite").save(OUTPUT_CONFIG)
print(f"✓ Configuration saved to: {OUTPUT_CONFIG}")

# Display the configuration
display(config_df.select("prefix", "priority", "pipeline_group", "source_schema", "tables", "schedule"))

# Step 3: Convert Spark DataFrame to Python for YAML Generation

In [0]:
# Collect data for YAML generation (small dataset, safe to collect)
config_data = config_df.collect()
print(f"Collected {len(config_data)} rows for YAML generation")

# Step 4: Define YAML Generation Functions

In [0]:
import yaml
from collections import defaultdict

def convert_cron_to_quartz(cron_expression: str) -> str:
    """Convert standard 5-field cron to Quartz 6-field format."""
    parts = cron_expression.strip().split()
    if len(parts) != 5:
        return cron_expression
    
    minute, hour, day, month, dow = parts
    if dow == '*':
        dow = '?'
    else:
        day = '?'
    
    return f"0 {minute} {hour} {day} {month} {dow}"

def generate_ga4_yaml(rows, output_path: str) -> None:
    """Generate Databricks Asset Bundle YAML for GA4 pipelines."""
    
    # Get default catalog and schema from first row
    default_catalog = rows[0]['target_catalog']
    default_schema = rows[0]['target_schema']
    
    print(f"\nTarget: {default_catalog}.{default_schema}")
    print(f"Total properties: {len(rows)}")
    
    # Group by pipeline_group
    groups = defaultdict(list)
    for row in rows:
        groups[row['pipeline_group']].append(row)
    
    print(f"Unique pipelines: {len(groups)}\n")
    
    # Build YAML
    combined_yaml = {
        "variables": {
            "dest_catalog": {"default": default_catalog},
            "dest_schema": {"default": default_schema}
        },
        "resources": {
            "pipelines": {},
            "jobs": {}
        }
    }
    
    # Generate pipelines
    for pipeline_group in sorted(groups.keys()):
        group_properties = groups[pipeline_group]
        pipeline_name = f"pipeline_ga4_{pipeline_group}"
        schedule = group_properties[0]['schedule']
        
        print(f"Pipeline: {pipeline_group}")
        print(f"  Properties: {len(group_properties)}")
        print(f"  Schedule: {schedule}")
        
        # Build ingestion objects
        ingestion_objects = []
        for row in group_properties:
            tables = [t.strip() for t in str(row['tables']).split(',')]
            for table in tables:
                ingestion_objects.append({
                    "table": {
                        "source_catalog": row['source_catalog'],
                        "source_schema": row['source_schema'],
                        "source_table": table,
                        "destination_catalog": "${var.dest_catalog}",
                        "destination_schema": "${var.dest_schema}",
                        "destination_table": f"{row['source_schema']}_{table}"
                    }
                })
            print(f"    - {row['source_schema']}: {', '.join(tables)}")
        
        # Add pipeline
        combined_yaml["resources"]["pipelines"][pipeline_name] = {
            "name": f"GA4 Ingestion - {pipeline_group}",
            "catalog": "${var.dest_catalog}",
            "ingestion_definition": {
                "connection_name": "ga4_connection",
                "objects": ingestion_objects
            }
        }
        
        # Add job
        job_name = f"job_ga4_{pipeline_group}"
        quartz_cron = convert_cron_to_quartz(schedule)
        combined_yaml["resources"]["jobs"][job_name] = {
            "name": f"GA4 Pipeline Scheduler - {pipeline_group}",
            "schedule": {
                "quartz_cron_expression": quartz_cron,
                "timezone_id": "UTC"
            },
            "tasks": [{
                "task_key": "run_ga4_pipeline",
                "pipeline_task": {
                    "pipeline_id": f"${{resources.pipelines.{pipeline_name}.id}}"
                }
            }]
        }
        print(f"  Job: {job_name} (Quartz: {quartz_cron})\n")
    
    # Write YAML to volume
    yaml_content = yaml.dump(combined_yaml, default_flow_style=False, sort_keys=False, indent=2)
    
    dbutils.fs.put(output_path, yaml_content, overwrite=True)
    
    print(f"✓ YAML generated: {output_path}")
    print(f"  Pipelines: {len(combined_yaml['resources']['pipelines'])}")
    print(f"  Jobs: {len(combined_yaml['resources']['jobs'])}")
    
    return yaml_content

print("Functions defined successfully")

# Step 5: Generate YAML

In [ ]:
# Configuration
OUTPUT_YAML = "/Volumes/your_catalog/your_schema/your_volume/yaml/ga4_pipeline.yml"

# Generate YAML
yaml_content = generate_ga4_yaml(config_data, OUTPUT_YAML)

# Results

In [0]:
# Display generated configuration summary
print("\nGenerated Pipeline Configuration Summary:")
print("="*80)
config_df.groupBy("pipeline_group", "schedule").agg(
    {"source_schema": "count"}
).orderBy("pipeline_group").show(truncate=False)

# Show YAML preview
print("\nGenerated YAML Preview:")
print("="*80)
print(yaml_content[:1500])
if len(yaml_content) > 1500:
    print("\n... (truncated)")
    print(f"\nFull YAML size: {len(yaml_content)} characters")

print("\n" + "="*80)
print("Files Generated:")
print("="*80)
print(f"1. Configuration: {OUTPUT_CONFIG}")
print(f"2. YAML: {OUTPUT_YAML}")

# Deploy using Databricks Asset Bundles CLI

Note: Run these commands in a terminal with the Databricks CLI configured